In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import os

paths = [
    "/content/drive/MyDrive/semicon",
    "/content/drive/MyDrive/semicon/generator5_augmented",
    "/content/drive/MyDrive/semicon/generator5_augmented/reference",
    "/content/drive/MyDrive/semicon/generator5_augmented/search",
    "/content/drive/MyDrive/semicon/generator5_augmented/training_metadata",
]

for p in paths:
    print(p, "->", os.path.exists(p))

/content/drive/MyDrive/semicon -> True
/content/drive/MyDrive/semicon/generator5_augmented -> True
/content/drive/MyDrive/semicon/generator5_augmented/reference -> True
/content/drive/MyDrive/semicon/generator5_augmented/search -> True
/content/drive/MyDrive/semicon/generator5_augmented/training_metadata -> True


In [7]:
!python /content/train_model_D.py \
    --epochs 2 \
    --batch-size 4 \
    --workers 2

Traceback (most recent call last):
  File "/content/train_model_D.py", line 9, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 442, in <module>
    from torch._C import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 463, in _lock_unlock_module
KeyboardInterrupt
^C


In [8]:
!python /content/train_model_D.py \
    --epochs 20 \
    --batch-size 4 \
    --workers 2

SEMICON / DRIFT-SENSE
MODEL D - SPATIAL CORRELATION LOCALIZATION
Device       : cuda
GPU          : Tesla T4
Input        : 1000 x 1000
Internal     : 128 x 128
Output       : spatial score map
Decoder      : soft-argmax
Epochs       : 20
Batch size   : 4

Trainable parameters: 338,369

EPOCH 1/20

Train loss       : 0.237384
Train mean error : 379.70 px
Val loss         : 0.202212
Val mean error   : 396.01 px
Val median error : 393.34 px
Val <= 5 px      : 0.00%
Val <= 10 px     : 0.00%
Epoch time       : 60.8 sec
Best checkpoint saved.

EPOCH 2/20

Train loss       : 0.192208
Train mean error : 373.76 px
Val loss         : 0.192685
Val mean error   : 391.64 px
Val median error : 406.63 px
Val <= 5 px      : 0.00%
Val <= 10 px     : 0.00%
Epoch time       : 60.0 sec
Best checkpoint saved.

EPOCH 3/20

Train loss       : 0.177176
Train mean error : 342.49 px
Val loss         : 0.185633
Val mean error   : 391.11 px
Val median error : 412.87 px
Val <= 5 px      : 0.00%
Val <= 10 px     :

In [4]:
!find "/content/drive/MyDrive/semicon/checkpoints" \
    -name "*.pt" \
    -type f

/content/drive/MyDrive/semicon/checkpoints/model_C/model_C_best.pt
/content/drive/MyDrive/semicon/checkpoints/model_C/model_C_last.pt
/content/drive/MyDrive/semicon/checkpoints/model_D/model_D_last.pt
/content/drive/MyDrive/semicon/checkpoints/model_D/model_D_best.pt
/content/drive/MyDrive/semicon/checkpoints/model_E_optimized/model_E_best.pt
/content/drive/MyDrive/semicon/checkpoints/model_E_optimized/model_E_last.pt
/content/drive/MyDrive/semicon/checkpoints/model_D_20epoch/model_D_best.pt
/content/drive/MyDrive/semicon/checkpoints/model_D_20epoch/model_D_last.pt


In [7]:
!python /content/evaluate_model_D.py

SEMICON / DRIFT-SENSE
FINAL TEST EVALUATION - MODEL D
Device     : cuda
GPU        : Tesla T4
Test CSV   : /content/drive/MyDrive/semicon/generator5_augmented/training_metadata/test.csv
Checkpoint : /content/drive/MyDrive/semicon/checkpoints/model_D/model_D_best.pt
Train code : /content/train_model_D.py

Loading original Model D architecture...
Model class: ModelD
Checkpoint loaded successfully.
Checkpoint epoch: 2
Checkpoint metrics: {'loss': 0.19159657898403348, 'mean_error': 387.0206604003906, 'median_error': 405.31744384765625, 'within_1': 0.0, 'within_5': 0.0, 'within_10': 0.0}

Test samples: 210
[  1/210] error=174.82 px
[ 25/210] error=456.02 px
[ 50/210] error=55.15 px
[ 75/210] error=162.82 px
[100/210] error=342.50 px
[125/210] error=547.92 px
[150/210] error=558.90 px
[175/210] error=392.21 px
[200/210] error=330.10 px
[210/210] error=410.23 px

MODEL D - FINAL TEST RESULT
Test samples       : 210
Mean error         : 396.210 px
Median error       : 409.434 px
Minimum error 

In [15]:
import shutil

src = "/content/drive/MyDrive/semicon/checkpoints/model_D/model_D_best.pt"
dst = "/content/drive/MyDrive/semicon_submission_model.pt"

shutil.copy2(src, dst)

print("Copied:", dst)

Copied: /content/drive/MyDrive/semicon_submission_model.pt


In [18]:
!python /create_30_test_cases.py

Test rows: 210
Prediction rows: 210
Prediction columns: ['index', 'reference_file', 'search_file', 'true_x', 'true_y', 'pred_x', 'pred_y', 'error_px']

DONE
Output folder: /content/drive/MyDrive/semicon/final_30_test_cases
30-case CSV: /content/drive/MyDrive/semicon/final_30_test_cases/selected_30_test_cases.csv
Worst-case image: /content/drive/MyDrive/semicon/final_30_test_cases/worst_case/worst_case.png
Worst-case explanation: /content/drive/MyDrive/semicon/final_30_test_cases/worst_case/explanation.txt


In [19]:
print(open(
    "/content/drive/MyDrive/semicon/final_30_test_cases/worst_case/explanation.txt"
).read())

SEMICON / DRIFT-SENSE - MODEL D WORST CASE

Test samples evaluated: 210
Reference file: 0021_aug_04_noise.png
Search file: 0021_aug_04_noise.png
Ground truth center: (142.5000, 928.5000) px
Predicted center: (722.1912, 318.3807) px
Localization error: 841.5981 px
Augmentation: noise
Target block: 3_0
Ambiguous: False

Visualization:
Green cross = ground-truth center
Red circle = predicted center
Magenta line = localization displacement



In [20]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/semicon/final_30_test_cases/selected_30_test_cases.csv"
)

print(df[[
    "reference_file",
    "search_file",
    "true_x",
    "true_y",
    "pred_x",
    "pred_y",
    "error_px"
]].to_string(index=False))

                     reference_file                         search_file   true_x   true_y     pred_x     pred_y   error_px
               0021_aug_05_blur.png                0021_aug_05_blur.png 643.5000 439.5000 631.494202 478.913971  41.201945
               0231_aug_05_blur.png                0231_aug_05_blur.png 382.5000 625.5000 616.799805 450.749329 292.291286
               0169_aug_05_blur.png                0169_aug_05_blur.png 331.5000 112.5000 706.968445 313.839600 426.044819
               0045_aug_05_blur.png                0045_aug_05_blur.png 403.5000  80.5000 638.427429 547.707947 522.947571
               0022_aug_05_blur.png                0022_aug_05_blur.png 125.5000 925.5000 509.259735 426.393005 629.586631
0218_aug_06_brightness_contrast.png 0218_aug_06_brightness_contrast.png 636.0000 380.0000 667.772278 305.086212  81.372927
0295_aug_06_brightness_contrast.png 0295_aug_06_brightness_contrast.png 836.5000 600.5000 688.874695 365.591583 277.444039
0018_aug_06_brig

In [21]:
import shutil

src = "/content/drive/MyDrive/semicon/final_30_test_cases"
dst = "/content/final_30_test_cases"

shutil.make_archive(dst, "zip", src)

print(dst + ".zip")

/content/final_30_test_cases.zip


In [22]:
from google.colab import files

files.download("/content/final_30_test_cases.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
import pandas as pd
import shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/semicon/generator5_augmented")
SELECTED = Path("/content/drive/MyDrive/semicon/final_30_test_cases/selected_30_test_cases.csv")

REF_DIR = BASE / "reference"
SEARCH_DIR = BASE / "search"

OUT = Path("/content/drive/MyDrive/semicon/final_30_test_cases_clean")
if OUT.exists():
    shutil.rmtree(OUT)

OUT.mkdir(parents=True)

df = pd.read_csv(SELECTED)

for i, row in df.iterrows():
    case_dir = OUT / f"case_{i+1:02d}"
    case_dir.mkdir()

    ref = REF_DIR / str(row["reference_file"])
    search = SEARCH_DIR / str(row["search_file"])

    shutil.copy2(ref, case_dir / "reference.png")
    shutil.copy2(search, case_dir / "search.png")

print("Created:", OUT)
print("Number of test cases:", len(df))

Created: /content/drive/MyDrive/semicon/final_30_test_cases_clean
Number of test cases: 30


In [24]:
from pathlib import Path

cases = sorted(Path(
    "/content/drive/MyDrive/semicon/final_30_test_cases_clean"
).glob("case_*"))

print("Cases:", len(cases))

for c in cases:
    print(c.name, "->",
          (c / "reference.png").exists(),
          (c / "search.png").exists())

Cases: 30
case_01 -> True True
case_02 -> True True
case_03 -> True True
case_04 -> True True
case_05 -> True True
case_06 -> True True
case_07 -> True True
case_08 -> True True
case_09 -> True True
case_10 -> True True
case_11 -> True True
case_12 -> True True
case_13 -> True True
case_14 -> True True
case_15 -> True True
case_16 -> True True
case_17 -> True True
case_18 -> True True
case_19 -> True True
case_20 -> True True
case_21 -> True True
case_22 -> True True
case_23 -> True True
case_24 -> True True
case_25 -> True True
case_26 -> True True
case_27 -> True True
case_28 -> True True
case_29 -> True True
case_30 -> True True


In [25]:
import shutil

shutil.make_archive(
    "/content/final_30_test_cases_clean",
    "zip",
    "/content/drive/MyDrive/semicon/final_30_test_cases_clean"
)

from google.colab import files
files.download("/content/final_30_test_cases_clean.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>